# corvia — voorbeeld: netwerkstromen reconstrueren uit sensortellingen

Deze notebook doorloopt de kern-workflow van `corvia` voor gesoten wegvak reconstructie van begin tot einde:

1. een wegennetwerk opbouwen uit een netwerkbestand
2. ruwe sensortellingen inladen en screenen in een `FlowStore`
3. consistente stromen over het volledige netwerk reconstrueren
4. nagaan hoe goed de reconstructie overeenkomt met de metingen
5. de resultaten exporteren en visualiseren

> **Opmerking:** deze notebook gebruikt placeholder-bestandsnamen onder
> `examples/data/`. Vervang deze door je eigen kleine voorbeelddataset (of
> een netwerk-/tellingenextract herleid tot enkele secties), zodat dit
> effectief end-to-end draait. Voer dit nooit uit op een volledige
> productiedataset — dat hoort thuis in een dedicated analyse-notebook, niet
> in een voorbeeld.

## 1. Setup

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from corvia.framework.builder import NetworkBuilder
from corvia.network_states.flow import FlowStore
from corvia.reconstruction.flow_link_balancer import FlowLinkBalancer
from corvia.engine.flow_resolver import FlowResolver
from corvia.engine.validators import ZScoreRejector, RelativeMarginAcceptor
from corvia.loaders.network import VCNetworkLoader
from corvia.loaders.flow_data_loader import VCFlowDataLoader

from corvia.logger import configure_logging
configure_logging(log_level="INFO", force_reconfigure=True)


## 2. Bouw het netwerk op

Een netwerk bestaat uit een verzameling van nodes die verbonden zijn door
directionele links. In `corvia` worden links gegroepeerd in `RoadSection`'s,
wanneer ze een continue weg vormen zonder splitsingen of kruisingen. Dit zijn
dus de eigenlijke wegvakken waarop stromen worden gereconstrueerd en gerapporteerd.
Daarnaast zijn er tellocaties (sensors) die verbonden zijn met de links. Zij vormen 
de connectie tussen de fysieke infrastructuur en de gemeten stromen.

Het `Network`-object houdt alle objecten van de volledige graaf bij en hun onderlinge 
relaties: links en nodes, de mapping van links in secties, connectie tussen links
en sensoren, ...



In dit voorbeeld wordt een netwerk ingelezen in het formaat van het 
Vlaamse Verkeerscentrum (VC). Hierbij werden de shapefiles van het VC 
samengevoegd tot een GeoPackage, zodat alle lagen in één bestand zitten:
- segmentdelen: de ruwe links van het netwerk
- segmenten: de gegroepeerde links in secties
- tellocaties: de sensoren die stromen meten

> **Merk op**: Het verkeerscentrum gebruikt dus geen nodes in hun netwerkbestanden,
> maar deze worden automatisch afgeleid uit de geometrie van de links.



`VCNetworkLoader` leest een VC-netwerk-GeoPackage in en geeft de ruwe links,
nodes en sensor-tellingslagen terug. `NetworkBuilder.from_data(...)` bouwt
hieruit een `Network`.

> We gebruiken hierbij de segmenten als links in de code om het aantal objecten
> te beperken. Een sectie kan meerdere links bevatten, maar in dit voorbeeld dus
> vaak slechts één link. De segmentdelen worden gebruikt door de loader om interne
> connecties te bepalen, maar worden niet opgenomen in het `Network`-object.



De `section_id_fn` hieronder geeft elke sectie de id van zijn eerste link,
in plaats van een automatisch gegenereerd `SECTION_0001`-nummer — puur om de
ids leesbaar te houden tijdens het inspecteren van resultaten.

In [ ]:
filename_network = "data/R2_network-VC.gpkg"

# Onderstaande laagnamen volgen de standaard Verkeerscentrum (VC)
# netwerkexportconventie — pas aan indien je bron andere laagnamen gebruikt.
loader = VCNetworkLoader(
    filename_network,
    links_layer="segmenten",
    parts_layer="segmentdelen",
    sensors_layer="locposten",
)
links_data, nodes_data, counts_data, crs = loader.load()

network = NetworkBuilder.from_data(
    name_network="voorbeeld-netwerk",
    links_data=links_data,
    nodes_data=nodes_data,
    counts_data=counts_data,
    crs=crs,
    section_id_fn=lambda chain: chain[0].link_id,
)

print(f"Netwerk opgebouwd: {len(network.links)} links, {len(network.sections)} secties")


## 3. Laad en screen de verkeersmetingen

`VCFlowDataLoader` leest ruwe sensortellingen in van het Vlaams Verkeerscentrum 
en koppelt deze, via `.prepare(network, ...)`, aan het netwerk. Hierbij worden 
de standaard voertuigklasses PW en VR gebruikt hun som TOTAL. Je kan hierbij ook
automatisch de metingen aggregeren tot een gewenste tijdsresolutie en de PAE
berekenen op basis van een opgegeven factor voor de VR-klasse.

Verder worden per meting kwaliteitsindicatoren berekend, zoals `pct_vol_reconstructed` 
(het aandeel van een telling dat de sensor zelf moest reconstrueren uit 
gedeeltelijke detecties). Je kan deze indicatoren opslaan in een reportbestand,
en een bestaand reportbestand inlezen om de screening te hergebruiken.

Op basis van deze indicatoren kennen we een **screening**-status toe per
meting:
- `held` — te veel van de telling is gereconstrueerd of foutief getypeerd om
  te vertrouwen
- `disputed` — grensgeval, wordt behouden maar gemarkeerd
- `accepted` — wordt as-is gebruikt

Deze screening gebeurt *vóór* de reconstructie op netwerkniveau — het is een
datakwaliteitscontrole op de ruwe sensorinput, los van de
rejector/acceptor-validatie die later tijdens de reconstructie wordt
toegepast.

In [ ]:
csv_filenames = ["data/R2_counts-VC_day.csv"]

flow_loader = VCFlowDataLoader(
    csv_filenames=csv_filenames,
    agg_freq="1D",
    pae_factor=2.0,
    start="2026-03-25",
    end="2026-03-26",
    vehicle_types=["TOTAL"],
    volume_err_fn=lambda df: df["volume"] * 0.005,
)

day_df = flow_loader.prepare(network, report="data/R2_sensor_report.csv")

conditions = [
    (day_df["pct_vol_reconstructed"] > 10.0) | (day_df["pct_vol_reco_typed"] > 5.0),
    (day_df["pct_vol_reconstructed"] >= 3.0)
    & (day_df["pct_vol_reconstructed"] <= 10.0)
    & (day_df["pct_vol_reco_typed"] <= 5.0),
]
choices = ["held", "disputed"]
day_df["screening"] = np.select(conditions, choices, default="accepted")

print("Screening-overzicht:")
print(day_df.drop_duplicates(subset=["source_id"])["screening"].value_counts())

store = FlowStore.from_dataframe(day_df)


## 4. Sanity check: komen de sensoren overeen met het netwerk?

Voor er iets wordt gereconstrueerd, is het de moeite waard om te
controleren of de sensor-ids in je tellingenbestand effectief overeenkomen
met sensorlocaties die in het netwerk zijn opgebouwd — verkeerde ids
(typfouten, een sensor die in het ene bestand staat maar niet in het
andere) duiken anders pas later op als "secties zonder metingen", zonder
dat de oorzaak duidelijk is.

In [ ]:
report_df = pd.read_csv("data/R2_sensor_report.csv", delimiter=";")
csv_sensors = set(report_df["LOCPOST"].astype(int).astype(str))

network_sensors = set()
for link in network.links.values():
    for sensor in link.sensor_locations:
        network_sensors.add(sensor.location_id)

matching = csv_sensors & network_sensors
only_in_csv = csv_sensors - network_sensors
only_in_network = network_sensors - csv_sensors

print(f"Overeenkomende sensoren:      {len(matching)}")
print(f"Enkel in tellingenbestand:    {len(only_in_csv)}")
print(f"Enkel in netwerk:             {len(only_in_network)}")


## 5. Validatie van voertuigstroommetingen

Reconstructie bestaat uit twee delen:

### 5.1 Identificeren van onbetrouwbare metingen (iteratief)

- **`FlowLinkBalancer`-reconstructors** bewaken het behoud van stroom
  tussen aangrenzende secties (wat binnenkomt moet ook buitengaan), zowel
  stroomopwaarts als stroomafwaarts en bij oplopende link-graad, zodat ook
  ijlere delen van het netwerk een fallback-schatting krijgen. Dit is het 
  gesloten wegvak principe dat we hier gebruiken om het volume van een link
  te schatten op basis van de metingen van zijn buren.
- **`Rejectors`** identificeren verdachte metingen die de reconstructie kunnen
  beïnvloeden. Niet alle metingen zijn even betrouwbaar of kwaliteitsvol waardoor
  er reconstructiefouten kunnen optreden. De rejector zoekt deze metingen op basis
  van de reconstructie uit de rest van het netwerk en markeert ze. In een volgende
  iteratie van het reconstructieproces worden deze metingen genegeerd, waardoor de
  reconstructie van de rest van het netwerk hopelijk verbeterd. De slechtste metingen
  worden eerst gemarkeerd totdat er een stabiele reconstructie is bereikt waarbij
  er geen metingen meer significant afwijken van de reconstructie. Er kunnen
  verschillende rejectors (of statistische testen) gebruikt worden. Standaard
  wordt de Z-score aangeraden.

`FlowResolver.run(...)` doorloopt dit proces iteratief (`max_iterations`)
over de opgegeven periodes en voertuigtypes.


### 5.2 Identificeren van betrouwbare metingen

Wanneer de reconstructie stabiel is, wordt een tweede validatiestap uitgevoerd om
te markeren welke metingen betrouwbaar lijken. Tijdens de reconstructie worden
metingen genegeerd die duidelijk afwijken, maar dat betekend niet dat ze daarom
betrouwbaar zijn. De **`Acceptor`** checkt of de metingen binnen een aanvaardbare marge
van de reconstructie liggen. Metingen die binnen deze marge vallen worden als
betrouwbaar beschouwd en kunnen gebruikt worden voor verdere analyses. Metingen die
buiten deze marge vallen worden als onopgelost beschouwd. Ze kunnen genegeerd worden
of verder onderzocht worden indien er te weinig gevalideerde metingen over blijven



In [ ]:
vehicle_type = "TOTAL"

reconstructors = [
    FlowLinkBalancer(direction="upstream", min_obs_degree=1, max_link_degree=3,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="upstream", min_obs_degree=2, max_link_degree=4,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="downstream", min_obs_degree=1, max_link_degree=3,
                      weight=1, allow_partial_fallback=True),
    FlowLinkBalancer(direction="downstream", min_obs_degree=2, max_link_degree=4,
                      weight=1, allow_partial_fallback=True),
]

resolver = FlowResolver(
    reconstructors=reconstructors,
    rejector=ZScoreRejector(use_errors=True, significance_level=0.10, median_mixing_fraction=0.),
    acceptor=RelativeMarginAcceptor(significance_level=0.05, max_percent_deviation=2.0, noise_floor=10., use_errors=True),
    max_iterations=30,
    name="voorbeeld-netwerk",
    greedy_elimination=True,
    greedy_adaptive_batch=True,
    greedy_k_max=1,
)

resolved_store = resolver.run(
    store=store,
    network=network,
    periods=[pd.Timestamp("2026-03-25")],
    vehicle_types=[vehicle_type],
)


## 6. Controleer de reconstructie-dekking

Een snel overzicht van hoeveel metingen per sectie `verified`, `rejected`
of `unresolved` zijn geworden — handig om secties op te sporen waar het
sensornetwerk te ijl is, of waar elke meting op een sectie als outlier werd
gemarkeerd (het waard om na te kijken vóór je het resultaat vertrouwt).

In [ ]:
obs_df = store.dataframe[store.dataframe["source_type"] == FlowStore.SOURCE_OBS].copy()

summary = obs_df.groupby("road_section_id").agg(
    total_obs=("validation", "count"),
    rejected_obs=("validation", lambda x: x.eq("rejected").sum()),
    verified_obs=("validation", lambda x: x.eq("verified").sum()),
    unresolved_obs=("validation", lambda x: x.eq("unresolved").sum()),
).reset_index()

network_section_ids = set(network.sections.keys())
observed_section_ids = set(summary["road_section_id"])
no_obs_sections = network_section_ids - observed_section_ids

print(f"Secties met metingen:        {len(summary)}")
print(f"Secties zonder metingen:     {len(no_obs_sections)}")
summary.head()


## 7. Vergelijk metingen met de herleide consensus

Voor elke ruwe meting koppelen we deze aan de herleide netwerkschatting
voor dezelfde sectie/tijdstip/voertuigtype en berekenen we het relatieve
verschil. Dit is de belangrijkste "klopt de reconstructie"-check —
systematisch grote afwijkingen bij één sensor wijzen eerder op een slecht
gekalibreerde of verkeerd geplaatste sensor dan op een netwerkprobleem.

In [ ]:
def get_sensor_vs_resolved(flow_df: pd.DataFrame) -> pd.DataFrame:
    """
    For every observation, pairs its own reading with the resolved consensus
    for its (road_section_id, timestamp, vehicle_type), and computes their
    relative difference with propagated error.

    Returns
    -------
    pd.DataFrame, indexed like the store's observation rows, with columns:
        source_id, road_section_id, timestamp, vehicle_type,
        volume, volume_err, screening, validation,
        resolved_volume, resolved_volume_err,
        relative_diff, relative_diff_err
    """
    df = flow_df.copy()

    obs_cols = ["source_id", "road_section_id", "timestamp", "vehicle_type",
                "volume", "volume_err", "screening", "validation"]
    obs_rows = df.loc[df["source_type"] == FlowStore.SOURCE_OBS, obs_cols].reset_index()

    resolved_rows = (
        df.loc[df["source_type"] == FlowStore.SOURCE_RES,
               ["road_section_id", "timestamp", "vehicle_type", "volume", "volume_err"]]
        .rename(columns={"volume": "resolved_volume", "volume_err": "resolved_volume_err"})
    )

    dupes = resolved_rows.duplicated(subset=["road_section_id", "timestamp", "vehicle_type"])
    if dupes.any():
        raise ValueError(f"Expected one resolved row per coordinate, found {int(dupes.sum())} duplicates.")

    merged = (
        obs_rows.merge(resolved_rows, on=["road_section_id", "timestamp", "vehicle_type"], how="left")
        .set_index("index")
    )
    merged.index.name = None

    merged["relative_diff"] = (merged["volume"] - merged["resolved_volume"]) / merged["resolved_volume"]
    merged["relative_diff_err"] = (
        np.sqrt(merged["volume_err"] ** 2 + merged["resolved_volume_err"] ** 2)
        / merged["resolved_volume"].abs()
    )

    merged["source_id"] = merged["source_id"].str.replace("sensor:", "", regex=True).astype(int)

    return merged


report = get_sensor_vs_resolved(resolved_store.dataframe)
report.to_csv(f"output/reconstruction_report_{vehicle_type}.csv", index=False)
report.head()


## 8. Exporteer de resultaten

Voeg het rapport samen met de sensorlocatie-laag, zodat resultaten
inspecteerbaar zijn in een GIS-tool naast het fysieke netwerk.

In [ ]:
sensors_df = gpd.read_file(filename_network, layer="locposten")
sensors_validated = sensors_df.merge(
    report, left_on="LOCPOST", right_on="source_id", how="left"
).drop(columns=["source_id"])

sensors_validated.to_file(
    "output/reconstruction_output.gpkg",
    layer=f"locposten_{vehicle_type}",
    driver="GPKG",
)


## 9. Visualiseer de sensorprestaties

Het relatieve verschil van elke sensor ten opzichte van de herleide
consensus, gekleurd naar validatiestatus, met gearceerde
aanvaardingsbanden ter referentie.

In [ ]:
df = sensors_validated.dropna(subset=["relative_diff"]).copy()
df = df.reindex(df["relative_diff"].abs().sort_values().index).reset_index(drop=True)

colours = {
    "rejected": "#d62728",
    "verified": "#2ca02c",
}
markers = {
    "rejected": "x",
    "verified": "o",
}

fig, ax = plt.subplots(figsize=(max(8, len(df) * 0.18), 6))
x = np.arange(len(df))
y = df["relative_diff"].abs() * 100
yerr = df["relative_diff_err"] * 100

seen = set()
for i, row in df.iterrows():
    status = row["validation"] if row["validation"] in colours else "unresolved"
    colour = colours.get(status, "#666666")
    marker = markers.get(status, "o")
    label = status if status not in seen else None
    ax.errorbar(x[i], y[i], yerr=yerr[i], fmt=marker, color=colour,
                capsize=3, elinewidth=1.5, zorder=3, label=label)
    seen.add(status)

ax.axhline(0, color="#1f77b4", alpha=0.3, zorder=1)
ax.axhspan(-2, 2, color="#E0E0E0", zorder=0, label="Aanvaardingsband (2%)")
ax.set_xticks(x)
ax.set_xticklabels(df["LOCPOST"], rotation=90, fontsize=6)
ax.set_yscale("symlog", linthresh=3, linscale=1.5)
ax.set_ylabel("Relatief verschil (%)")
ax.set_xlabel("Sensor (gesorteerd op relatief verschil)")
ax.set_title(f"Sensor vs. herleide consensus — {vehicle_type}")
ax.legend(loc="best", framealpha=0.9)
fig.tight_layout()
fig.savefig(f"output/{vehicle_type}_sensor_performance.png", dpi=200)


## Vervolgstappen

- Vervang de netwerk-GeoPackage en tellingen-CSV onder `examples/data/`
  door je eigen data.
- Zie de README van de package voor een overzicht van de kernconcepten die
  hier gebruikt worden (`Network`, `FlowStore`, reconstructors,
  rejector/acceptor).
- Voor meer detail over het afstellen van de rejector/acceptor-drempels,
  zie de API-referentie van `corvia.engine.validators`.

> Aannames die het waard zijn om na te kijken tegenover de actuele
> codebase: de exacte keyword-argumenten die `VCFlowDataLoader.prepare()`
> en `FlowResolver.run()` aanvaarden, en of `segmenten_aangepast` /
> `segmentdelen_aangepast` / `locposten_aangepast` een vaste
> VC-exportconventie zijn of specifiek voor individuele studies.